# Imports and Database Connection

In [11]:
import re
from mongoengine import connect, disconnect
from pycoshark.mongomodels import Project, VCSSystem, Commit, FileAction, Hunk, Refactoring, IssueSystem, Issue, IssueComment, MailingList, Message
from pycoshark.utils import create_mongodb_uri_string


# You may have to update this dict to match your DB credentials
credentials = {'db_user': '',
               'db_password': '',
               'db_hostname': 'localhost',
               'db_port': 27017,
               'db_authentication_database': '',
               'db_ssl_enabled': False}

uri = create_mongodb_uri_string(**credentials)

disconnect(alias='default')

connect('smartshark_2_1', host=uri, alias='default')

MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True, read_preference=Primary())

# Working with issues

In [12]:
from pycoshark.mongomodels import Project, VCSSystem, Commit, FileAction, Hunk, Refactoring, IssueSystem, Issue, IssueComment, MailingList, Message

# 1. Safely retrieve the project. Using .first() prevents a crash if there are duplicate project names.
project_name = 'giraph'
project = Project.objects(name=project_name).first()

if not project:
    print(f"CRITICAL: Project '{project_name}' not found in the database.")
else:
    # 2. Fix the tracker crash. We extract all trackers, but explicitly select the first one to proceed.
    issue_trackers = IssueSystem.objects(project_id=project.id)
    
    if issue_trackers.count() == 0:
        print(f"No issue trackers found for {project_name}.")
    else:
        # Safely grab the primary tracker
        issue_tracker = issue_trackers.first()
        print(f"Issue Tracker Selected: {issue_tracker.url} (1 of {issue_trackers.count()} trackers found)")

        # 3. Count total issues
        num_issues = Issue.objects(issue_system_id=issue_tracker.id).count()
        print(f"Total issues to process: {num_issues}")

        count_comments = 0
        count_referenced_by_commits = 0
        count_bugs_dev_label = 0
        count_bugs_validated = 0

        print("Executing N+1 query loop. This will take several minutes...")
        
        # 4. Execute the loop with a progress tracker
        issues = Issue.objects(issue_system_id=issue_tracker.id)
        for i, issue in enumerate(issues):
            # Print a progress update every 500 issues so you know it hasn't crashed
            if i % 500 == 0 and i > 0:
                print(f"... Processed {i} / {num_issues} issues ...")
                
            count_comments += IssueComment.objects(issue_id=issue.id).count()
            
            if issue.issue_type is not None and issue.issue_type.lower() == 'bug':
                count_bugs_dev_label += 1
                
            if issue.issue_type_verified is not None and issue.issue_type_verified.lower() == 'bug':
                count_bugs_validated += 1
                
            if Commit.objects(linked_issue_ids=issue.id).count() > 0:
                count_referenced_by_commits += 1
                
        print("\n--- Execution Complete ---")
        print('Number of comments in discussions:', count_comments)
        print('Number of issues referenced by commits:', count_referenced_by_commits)
        print('Number of issues labeled as bugs by developers:', count_bugs_dev_label)
        print('Number of issues labeled validated as bug by researchers:', count_bugs_validated)

Issue Tracker Selected: https://issues.apache.org/jira/rest/api/2/search?jql=project=GIRAPH (1 of 1 trackers found)
Total issues to process: 1232
Executing N+1 query loop. This will take several minutes...
... Processed 500 / 1232 issues ...
... Processed 1000 / 1232 issues ...

--- Execution Complete ---
Number of comments in discussions: 6759
Number of issues referenced by commits: 787
Number of issues labeled as bugs by developers: 534
Number of issues labeled validated as bug by researchers: 140
